<a href="https://colab.research.google.com/github/suneel-ratnala/Agentic_AI_Practice/blob/main/day1/LCEL/lcel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### LCEL Deepdive

Install required dependencies for LangChain, Groq API integration, and vector storage.

In [1]:
!pip install -qU langchain-groq langchain_community langchain_huggingface faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


Set up environment variables and import necessary LangChain components for building chains with the Groq LLM.

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
#from dotenv import load_dotenv
#load_dotenv()
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

Create a simple LCEL chain: prompt → model → output parser. This demonstrates the pipe operator (|) chaining components together.

In [3]:
prompt = ChatPromptTemplate.from_template("Explain about {topic} in detail")
#model = ChatOpenAI()
# Initialize the Groq model
model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_retries=2,
)
output_parser = StrOutputParser()

chain = prompt | model | output_parser

chain.invoke({"topic": "ice cream"})

'Ice cream is a frozen dessert made from a mixture of cream, sugar, and flavorings, typically frozen to a smooth and creamy consistency. It has been a popular treat for centuries, with its origins dating back to ancient civilizations in the Middle East and China.\n\n**History of Ice Cream**\n\nThe earliest known evidence of ice cream-like desserts dates back to ancient Mesopotamia, around 2000 BC. The ancient Greeks and Romans also enjoyed frozen desserts made from snow and sweetened with honey. However, the modern version of ice cream as we know it today originated in Italy in the 16th century.\n\nThe Medici family in Florence, Italy, commissioned a chef named Bernardo Buontalenti to create a frozen dessert for a banquet. Buontalenti created a frozen mixture of cream, sugar, and fruit, which became known as "gelato." The name "gelato" is still used today to describe Italian-style ice cream.\n\n**Ingredients of Ice Cream**\n\nIce cream is typically made from a combination of the follow

Inspect the prompt template output to see how variables are formatted before being passed to the model.

In [4]:
print(prompt.invoke({"topic": "ice cream"}))

messages=[HumanMessage(content='Explain about ice cream in detail', additional_kwargs={}, response_metadata={})]


Invoke the model directly with chat messages to generate a response.

In [5]:
from langchain_core.messages.human import HumanMessage

messages = [HumanMessage(content='tell me a short joke about ice cream')]
model.invoke(messages)

AIMessage(content='Why did the ice cream go to the doctor? \n\nBecause it had a meltdown.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 43, 'total_tokens': 61, 'completion_time': 0.025632996, 'completion_tokens_details': None, 'prompt_time': 0.002134551, 'prompt_tokens_details': None, 'queue_time': 0.046758979, 'total_time': 0.027767547}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf165-1d24-7531-b581-5281b4d6cac3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 18, 'total_tokens': 61})

### What is this "|" in Python?

Implement a custom Runnable class to understand how the pipe operator (|) works. This creates a chain pattern where each component processes data sequentially.

In [6]:
from abc import ABC, abstractmethod

class CRunnable(ABC):
    def __init__(self):
        self.next = None

    @abstractmethod
    def process(self, data):
        """
        This method must be implemented by subclasses to define
        data processing behavior.
        """
        pass

    def invoke(self, data):
        processed_data = self.process(data)
        if self.next is not None:
            return self.next.invoke(processed_data)
        return processed_data

    def __or__(self, other):
        return CRunnableSequence(self, other)

class CRunnableSequence(CRunnable):
    def __init__(self, first, second):
        super().__init__()
        self.first = first
        self.second = second

    def process(self, data):
        return data

    def invoke(self, data):
        first_result = self.first.invoke(data)
        return self.second.invoke(first_result)



Define concrete Runnable implementations (AddTen, MultiplyByTwo, ConvertToString) to demonstrate data transformation through a chain.

In [7]:
class AddTen(CRunnable):
    def process(self, data):
        print("AddTen: ", data)
        return data + 10

class MultiplyByTwo(CRunnable):
    def process(self, data):
        print("Multiply by 2: ", data)
        return data * 2

class ConvertToString(CRunnable):
    def process(self, data):
        print("Convert to string: ", data)
        return f"Result: {data}"

Create a chain by composing three custom Runnables using the pipe operator.

In [8]:
a = AddTen()
b = MultiplyByTwo()
c = ConvertToString()

chain = a | b | c

Execute the chain and observe how data flows through each component sequentially.

In [9]:
result = chain.invoke(10)
print(result)

AddTen:  10
Multiply by 2:  20
Convert to string:  40
Result: 40


### Runnables from LangChain

Import LangChain's built-in Runnable types: RunnablePassthrough (passes data through unchanged), RunnableLambda (wraps functions), and RunnableParallel (processes multiple branches).

In [10]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

Chain multiple RunnablePassthrough instances together, demonstrating that passthrough components return data unchanged.

In [11]:
chain = RunnablePassthrough() | RunnablePassthrough () | RunnablePassthrough ()
chain.invoke("hello")

'hello'

Define a lambda function that converts input strings to uppercase.

In [12]:
def input_to_upper(input: str):
    output = input.upper()
    return output

Chain RunnablePassthrough with RunnableLambda to apply the uppercase transformation.

In [13]:
chain = RunnablePassthrough() | RunnableLambda(input_to_upper) | RunnablePassthrough()
chain.invoke("hello")

'HELLO'

Create a RunnableParallel that simultaneously processes the same input through multiple branches, each stored as a key in the output dictionary.

In [14]:
chain = RunnableParallel({"x": RunnablePassthrough(), "y": RunnablePassthrough()})

Execute the parallel chain with a simple string input.

In [15]:
chain.invoke("hello")

{'x': 'hello', 'y': 'hello'}

Execute the parallel chain with a dictionary input to see how RunnablePassthrough handles structured data.

In [16]:
chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': {'input': 'hello', 'input2': 'goodbye'},
 'y': {'input': 'hello', 'input2': 'goodbye'}}

Create a RunnableParallel with a lambda function that extracts a specific key from the input dictionary.

In [17]:
chain = RunnableParallel({"x": RunnablePassthrough(), "y": lambda z: z["input2"]})

Execute the parallel chain, demonstrating selective extraction of input values.

In [18]:
chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': {'input': 'hello', 'input2': 'goodbye'}, 'y': 'goodbye'}

### Nested chains - now it gets more complicated!

Define a lambda function that extracts and transforms a specific key from the input dictionary.

In [19]:
def find_keys_to_uppercase(input: dict):
    output = input.get("input", "not found").upper()
    return output

Create a complex nested chain with RunnableParallel containing a sequential pipe (RunnablePassthrough | RunnableLambda) in one branch and a lambda in another.

In [20]:
chain = RunnableParallel({"x": RunnablePassthrough() | RunnableLambda(find_keys_to_uppercase), "y": lambda z: z["input2"]})

Execute the nested chain to show how complex transformations combine parallel and sequential processing.

In [21]:
chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': 'HELLO', 'y': 'goodbye'}

Define helper functions for assignment and multiplication, then create a RunnableParallel as a base for further chaining.

In [22]:
chain = RunnableParallel({"x": RunnablePassthrough()})

def assign_func(input):
    return 100

def multiply(input):
    return input * 10

Execute the parallel chain with dictionary input.

In [23]:
chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': {'input': 'hello', 'input2': 'goodbye'}}

Use the .assign() method to add new computed fields to the chain output while preserving existing data.

In [24]:
chain = RunnableParallel({"x": RunnablePassthrough()}).assign(extra=RunnableLambda(assign_func))

Execute the chain with assignment to display the combined result with both original and computed values.

In [25]:
result = chain.invoke({"input": "hello", "input2": "goodbye"})
print(result)

{'x': {'input': 'hello', 'input2': 'goodbye'}, 'extra': 100}


### Combine multiple chains (incl. coercion)

Create extraction and transformation functions, then compose them into a new chain that extracts data and applies uppercase conversion.

In [26]:
def extractor(input: dict):
    return input.get("extra", "Key not found")

def cupper(upper: str):
    return str(upper).upper()

new_chain = RunnableLambda(extractor) | RunnableLambda(cupper)

Execute the extraction and transformation chain on a dictionary input.

In [27]:
new_chain.invoke({"extra": "test"})

'TEST'

Combine the complex nested chain with the extraction chain to demonstrate chain coercion and composition.

In [28]:
final_chain = chain | new_chain
final_chain.invoke({"input": "hello", "input2": "goodbye"})

'100'

### Real World Example

Build a Retrieval-Augmented Generation (RAG) chain: create embeddings, initialize a vector store with sample data, set up a retriever, and chain it with a prompt and LLM for context-aware question answering.

In [29]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
#from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_texts(
    ["Cats love thuna"], embedding=embeddings
)
retriever = vectorstore.as_retriever()
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template=template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    RunnableParallel({"context": retriever | format_docs, "question": RunnablePassthrough()})
    | prompt
    | ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
    | StrOutputParser()
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Query the RAG chain with a question to generate an answer based on retrieved context.

In [30]:
rag_chain.invoke("What do cats like to eat?")

'Cats love thuna.'

Debug the RAG chain by inspecting the output of the RunnableParallel component to see the retrieved context and question.

In [31]:
RunnableParallel({"context": retriever | format_docs, "question": RunnablePassthrough()}).invoke("What do cats like to eat?")

{'context': 'Cats love thuna', 'question': 'What do cats like to eat?'}

Inspect the formatted prompt with injected context and question values.

In [32]:
prompt.invoke({"context": "Cats love thuna", "question": "What do cats like to eat?"})

ChatPromptValue(messages=[HumanMessage(content='Answer the question based only on the following context:\nCats love thuna\n\nQuestion: What do cats like to eat?\n', additional_kwargs={}, response_metadata={})])

Execute the model on the formatted prompt to see the final LLM response before parsing.

In [33]:
model.invoke(prompt.invoke({"context": "Cats love thuna", "question": "What do cats like to eat?"}))

AIMessage(content='Cats love thuna.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 60, 'total_tokens': 67, 'completion_time': 0.017086754, 'completion_tokens_details': None, 'prompt_time': 0.002947755, 'prompt_tokens_details': None, 'queue_time': 0.047052725, 'total_time': 0.020034509}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf17b-447a-7910-8a71-648a2b41f8c7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 7, 'total_tokens': 67})